In [ ]:
# --- CELL 1: Setup ---
import os
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate



C:\Users\Deepak\AppData\Local\Temp\ipykernel_10744\2614247146.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:
# --- CELL 2: Fetch Transcript ---
video_id = "Gfr50f6ZBvo"

try:
    # Fetch the transcript
    ytt_api = YouTubeTranscriptApi()
    transcript_list = ytt_api.fetch(video_id, languages=["en"])

    # FIX: Use chunk.text instead of chunk["text"]
    transcript = " ".join(chunk.text for chunk in transcript_list)

    print("Transcript successfully fetched!")

    # Print the first 500 characters to verify it worked
    print("\nPreview:\n", transcript[:500], "...")

except TranscriptsDisabled:
    print("No captions are available for this video.")
except Exception as e:
    print(f"An error occurred: {e}")

Transcript successfully fetched!

Preview:
 the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful  ...


In [ ]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=splitter.create_documents([transcript])

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Initialize the free Hugging Face embedding model (it will download the first time you run it)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create your FAISS vector store
vector_store = FAISS.from_documents(chunks, embeddings)

print("Boom! Vector store created successfully for exactly $0.00!")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 1607.54it/s]


Boom! Vector store created successfully for exactly $0.00!


In [ ]:
retriever=vector_store.as_retriever(search_type="similarity",search_kwargs={"k":4})

In [ ]:
retriever.invoke("What is Deepmind")

[Document(id='6a33f26b-ace7-4fbd-ba5a-784f9c6537ec', metadata={}, page_content="and how it works this is tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware compute infrastructure how much is it the software computer infrastructure yeah um what else is there how much is the human infrastructure and like just the humans interact in certain kinds of ways in all the space of all those ideas how much does maybe like philosophy how much what's the key if um uh if if you were to sort of look back like if we go forward 200 years look back what was the key 

In [ ]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
# FIX: Import from langchain_classic instead of langchain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.2)
os.environ["GROQ_API_KEY"]="YOUR_API_KEY"

In [ ]:
# 1. Define the system prompt string
system_prompt = (
    "You are a helpful assistant. Use the following pieces of retrieved video "
    "transcript to answer the question. If you don't know the answer, just say "
    "that you don't know. \n\n"
    "Context: {context}"
)

# 2. Create the ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 3. Create the document combination chain & final retrieval chain
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

print("RAG Chain created successfully!")

RAG Chain created successfully!


In [ ]:
from langchain_core.runnables import RunnableParallel,RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser


In [ ]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),#when the context arrives it goes to retriever and it goes to format_docs
    'input': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke('who is demis')

{'context': "to get world peace because there's also other corrupting things like wanting power over people and this kind of stuff which is not necessarily satisfied by by just abundance but i think it will help um and i think uh but i think ultimately ai is not going to be run by any one person or one organization i think it should belong to the world belong to humanity um and i think maybe many there'll be many ways this will happen and ultimately um everybody should have a say in that do you have advice for uh young people in high school and college maybe um if they're interested in ai or interested in having a big impact on the world what they should do to have a career they can be proud of her to have a life they can be proud of i love giving talks to the next generation what i say to them is actually two things i i think the most important things to learn about and to find out about when you're when you're young is what are your true passions is first of all there's two things on

In [ ]:
parser=StrOutputParser()
main_chain=parallel_chain | prompt | llm |parser
main_chain.invoke("can you summarise the video")

"The video transcript appears to be a conversation between the host, Demas, and a guest (not explicitly mentioned). They discuss various topics, including:\n\n1. Explaining complex concepts: The guest mentions that a deeper understanding of physics might involve simpler explanations, and that this could encompass mysteries like consciousness, life, and gravity.\n2. Intelligence and explanation: The guest notes that the ability to explain complex topics simply is a sign of intelligence, citing Richard Feynman as an example.\n3. Human-AI symbiosis: The guest discusses how humans are already symbiotic with their compute devices, and that this relationship could be enhanced in the future.\n4. History of AI: The guest mentions Claude Shannon's first chess program (1949) and Alan Turing's chess program, which was run manually due to the slow computers at the time.\n5. Deep Blue and Kasparov: The guest recalls the 1997 chess match between IBM's Deep Blue and Garry Kasparov, and notes that he 